CLEANING THE DATSET GIVEN FOR THE ANALYSIS


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import FunctionTransformer


CLEANING


In [3]:

# Correct folder path (because notebook is already inside Token-Trend)
folder = Path("DATA")
files = sorted([f for f in folder.glob("*.csv") if not f.name.startswith("cleaned_")])

required_cols = ["date", "open", "high", "low", "close", "volume"]
numeric_cols = ["open", "high", "low", "close", "volume"]

summary = []

for file_path in files:
    raw = pd.read_csv(file_path)
    original_rows = len(raw)

    # Standardize column names
    raw.columns = raw.columns.str.strip().str.lower()

    # Check required columns
    missing_cols = [c for c in required_cols if c not in raw.columns]
    if missing_cols:
        summary.append({
            "file": file_path.name,
            "status": f"skipped (missing columns: {missing_cols})",
            "original_rows": original_rows,
            "cleaned_rows": 0,
            "rows_removed": original_rows
        })
        continue

    # Clean data directly on raw
    raw = raw.dropna(subset=required_cols)
    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
    for c in numeric_cols:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
    raw = raw.dropna(subset=required_cols)

    # Business rules
    raw = raw[
        (raw["open"] >= 0) &
        (raw["high"] >= 0) &
        (raw["low"] >= 0) &
        (raw["close"] >= 0) &
        (raw["volume"] >= 0) &
        (raw["high"] >= raw["low"])
    ].drop_duplicates()

    cleaned_rows = len(raw)
    removed_rows = original_rows - cleaned_rows

    # Save with changed file name: cleaned_<original>.csv
    cleaned_name = f"cleaned_{file_path.name}"
    cleaned_path = file_path.with_name(cleaned_name)
    raw.to_csv(cleaned_path, index=False)

    summary.append({
        "file": file_path.name,
        "new_file": cleaned_name,
        "status": "cleaned and saved as new filename",
        "original_rows": original_rows,
        "cleaned_rows": cleaned_rows,
        "rows_removed": removed_rows
    })

summary_df = pd.DataFrame(summary)
summary_df

,file,status,original_rows,cleaned_rows,rows_removed
0,file_importance_scores.csv,"skipped (missing columns: ['date', 'open', 'hi...",23,0,23


LOGISTIC REGRESSION 

FILE cleaned_coin_Bitcoin.csv
only logistic regresssion with normal feateures]

In [4]:
# Logistic Regression for ONE file only with linear terms
# Target: 1 if next day's Close > today's Close, else 0
# Model form per feature: a1*x (no sqrt, no x^2, no interactions)
# Train/Test: first 3/4 rows for training, last 1/4 rows for testing

# -----------------------------
# 1) Select one file
# -----------------------------
file_path = "DATA/cleaned_coin_Bitcoin.csv"
df = pd.read_csv(file_path)

# -----------------------------
# 2) Helper for flexible column matching
# -----------------------------
def _norm(s: str):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def find_col(columns, candidates):
    norm_map = {_norm(c): c for c in columns}
    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]
    return None

open_col  = find_col(df.columns, ["open", "priceopen"])
high_col  = find_col(df.columns, ["high", "pricehigh"])
low_col   = find_col(df.columns, ["low", "pricelow"])
close_col = find_col(df.columns, ["close", "priceclose"])
vol_col   = find_col(df.columns, ["volume", "volumefrom", "volumeto", "totalvolume", "tradevolume"])

if any(c is None for c in [open_col, high_col, low_col, close_col, vol_col]):
    raise ValueError("Required columns not found (need open/high/low/close/volume).")

# -----------------------------
# 3) Build base features and target
# -----------------------------
base_features = ["open", "high", "low", "close", "volume"]
data = df[[open_col, high_col, low_col, close_col, vol_col]].copy()
data.columns = base_features
data = data.apply(pd.to_numeric, errors="coerce")

# Target: next-day movement of close
data["target_up_next_day"] = (data["close"].shift(-1) > data["close"]).astype(int)
data = data.dropna().reset_index(drop=True)

X = data[base_features]
y = data["target_up_next_day"].astype(int)

# -----------------------------
# 4) 3/4 split (time order)
# -----------------------------
n = len(X)
split_idx = (3 * n) // 4
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# -----------------------------
# 5) Train model
# -----------------------------
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, C=10.0, random_state=42))
])
model.fit(X_train, y_train)

# -----------------------------
# 6) Accuracy
# -----------------------------
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

accuracy_df = pd.DataFrame([{
    "file": file_path,
    "rows_used": n,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "accuracy": accuracy
}])

# -----------------------------
# 7) Extract linear weights
# -----------------------------
clf = model.named_steps["clf"]
coef = clf.coef_.ravel()
intercept = clf.intercept_[0]
coef_map = dict(zip(base_features, coef))

linear_weight_df = pd.DataFrame({
    "file": file_path,
    "feature": base_features,
    "a1_linear": [coef_map[f] for f in base_features]
})
linear_weight_df["equation"] = linear_weight_df.apply(
    lambda r: f"{r['feature']}: {r['a1_linear']:.6f}*x",
    axis=1
)

degree_wise_weights_df = pd.DataFrame({
    "file": file_path,
    "feature": base_features,
    "degree": [1.0] * len(base_features),
    "term": base_features,
    "weight": [coef_map[f] for f in base_features]
})

# -----------------------------
# 8) Feature importance (linear only)
# -----------------------------
abs_coef = np.abs(coef)
importance_pct = np.zeros_like(abs_coef) if abs_coef.sum() == 0 else (abs_coef / abs_coef.sum()) * 100

feature_importance_df = pd.DataFrame({
    "file": file_path,
    "feature": base_features,
    "abs_weight": abs_coef,
    "importance_score_percent": importance_pct
}).sort_values("importance_score_percent", ascending=False).reset_index(drop=True)
feature_importance_df["rank"] = np.arange(1, len(feature_importance_df) + 1)
feature_importance_df = feature_importance_df[[
    "file", "rank", "feature", "abs_weight", "importance_score_percent"
]]

# -----------------------------
# 9) Display outputs
# -----------------------------
print("Accuracy:")
print(accuracy_df.to_string(index=False))

print("\nPer-feature linear weights (a1*x):")
print(linear_weight_df.to_string(index=False))

print("\nDegree-wise weights table:")
print(degree_wise_weights_df.to_string(index=False))

print("\nFeature Importance Scores:")
print(feature_importance_df.to_string(index=False))

Accuracy:
                         file  rows_used  train_rows  test_rows  accuracy
DATA/cleaned_coin_Bitcoin.csv       2991        2243        748  0.528075

Per-feature linear weights (a1*x):
                         file feature  a1_linear           equation
DATA/cleaned_coin_Bitcoin.csv    open   0.309711   open: 0.309711*x
DATA/cleaned_coin_Bitcoin.csv    high   0.015150   high: 0.015150*x
DATA/cleaned_coin_Bitcoin.csv     low  -0.143575   low: -0.143575*x
DATA/cleaned_coin_Bitcoin.csv   close  -0.249485 close: -0.249485*x
DATA/cleaned_coin_Bitcoin.csv  volume   0.067107 volume: 0.067107*x

Degree-wise weights table:
                         file feature  degree   term    weight
DATA/cleaned_coin_Bitcoin.csv    open     1.0   open  0.309711
DATA/cleaned_coin_Bitcoin.csv    high     1.0   high  0.015150
DATA/cleaned_coin_Bitcoin.csv     low     1.0    low -0.143575
DATA/cleaned_coin_Bitcoin.csv   close     1.0  close -0.249485
DATA/cleaned_coin_Bitcoin.csv  volume     1.0 volume  0

    NEW FILE+ FEATURES

In [ ]:
# Logistic Regression for ONE file with improvements:
# 1) Time-series cross-validation + hyperparameter tuning
# 2) Stationary features (returns/ranges/changes/volatility)
# 4) Stronger target definition using 2% next-day return threshold

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# -----------------------------
# 1) Select one file
# -----------------------------
file_path = "DATA/cleaned_coin_Bitcoin.csv"
df = pd.read_csv(file_path)

# -----------------------------
# 2) Helper for flexible column matching
# -----------------------------
def _norm(s: str):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def find_col(columns, candidates):
    norm_map = {_norm(c): c for c in columns}
    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]
    return None

open_col = find_col(df.columns, ["open", "priceopen"])
high_col = find_col(df.columns, ["high", "pricehigh"])
low_col = find_col(df.columns, ["low", "pricelow"])
close_col = find_col(df.columns, ["close", "priceclose"])
vol_col = find_col(df.columns, ["volume", "volumefrom", "volumeto", "totalvolume", "tradevolume"])

if any(c is None for c in [open_col, high_col, low_col, close_col, vol_col]):
    raise ValueError("Required columns not found (need open/high/low/close/volume).")

# -----------------------------
# 3) Build stationary features + 2% threshold target
# -----------------------------
data = df[[open_col, high_col, low_col, close_col, vol_col]].copy()
data.columns = ["open", "high", "low", "close", "volume"]
data = data.apply(pd.to_numeric, errors="coerce")

# Stationary features
# Returns/momentum style
data["ret_1d"] = data["close"].pct_change(1)
data["ret_3d"] = data["close"].pct_change(3)

# Relative candle/range features
data["hl_range_pct"] = (data["high"] - data["low"]) / data["close"].replace(0, np.nan)
data["co_change_pct"] = (data["close"] - data["open"]) / data["open"].replace(0, np.nan)

# Volatility + volume change features
data["volatility_5"] = data["ret_1d"].rolling(5).std()
data["volatility_10"] = data["ret_1d"].rolling(10).std()
data["volume_chg_1d"] = data["volume"].pct_change(1)

# Trend ratio as relative value (still stationary-like)
data["sma_5"] = data["close"].rolling(5).mean()
data["sma_10"] = data["close"].rolling(10).mean()
data["trend_sma_ratio"] = (data["sma_5"] / data["sma_10"]) - 1

# Target with threshold = 2%
# 1 if next-day return > 2%, else 0
threshold = 0.02
data["next_day_return"] = data["close"].shift(-1) / data["close"] - 1
data["target_up_next_day"] = (data["next_day_return"] > threshold).astype(int)

feature_cols = [
    "ret_1d", "ret_3d", "hl_range_pct", "co_change_pct",
    "volatility_5", "volatility_10", "volume_chg_1d", "trend_sma_ratio"
]

data = data.dropna(subset=feature_cols + ["target_up_next_day"]).reset_index(drop=True)

X = data[feature_cols]
y = data["target_up_next_day"].astype(int)

# -----------------------------
# 4) Time-aware train/test split (3/4, 1/4)
# -----------------------------
n = len(X)
split_idx = (3 * n) // 4
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# -----------------------------
# 5) Time-series CV tuning on train set
# -----------------------------
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=6000, random_state=42))
])

param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0, 50.0],
    "clf__penalty": ["l1", "l2"],
    "clf__solver": ["liblinear"],
    "clf__class_weight": [None, "balanced"]
}

tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=tscv,
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# -----------------------------
# 6) Summaries
# -----------------------------
accuracy_df = pd.DataFrame([{
    "file": file_path,
    "rows_used": n,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "target_threshold_pct": threshold * 100,
    "positive_rate_pct": y.mean() * 100,
    "best_cv_accuracy": grid.best_score_,
    "test_accuracy": accuracy
}])

clf = best_model.named_steps["clf"]
coef = clf.coef_.ravel()
intercept = clf.intercept_[0]
coef_map = dict(zip(feature_cols, coef))

linear_weight_df = pd.DataFrame({
    "file": file_path,
    "feature": feature_cols,
    "a1_linear": [coef_map[f] for f in feature_cols]
})
linear_weight_df["equation"] = linear_weight_df.apply(
    lambda r: f"{r['feature']}: {r['a1_linear']:.6f}*x",
    axis=1
)

degree_wise_weights_df = pd.DataFrame({
    "file": file_path,
    "feature": feature_cols,
    "degree": [1.0] * len(feature_cols),
    "term": feature_cols,
    "weight": [coef_map[f] for f in feature_cols]
})

abs_coef = np.abs(coef)
importance_pct = np.zeros_like(abs_coef) if abs_coef.sum() == 0 else (abs_coef / abs_coef.sum()) * 100

feature_importance_df = pd.DataFrame({
    "file": file_path,
    "feature": feature_cols,
    "abs_weight": abs_coef,
    "importance_score_percent": importance_pct
}).sort_values("importance_score_percent", ascending=False).reset_index(drop=True)
feature_importance_df["rank"] = np.arange(1, len(feature_importance_df) + 1)
feature_importance_df = feature_importance_df[[
    "file", "rank", "feature", "abs_weight", "importance_score_percent"
]]

print("Best hyperparameters:")
print(grid.best_params_)

print("\nAccuracy Summary:")
print(accuracy_df.to_string(index=False))

print("\nPer-feature linear weights (a1*x):")
print(linear_weight_df.to_string(index=False))

print("\nDegree-wise weights table:")
print(degree_wise_weights_df.to_string(index=False))

print("\nFeature Importance Scores:")
print(feature_importance_df.to_string(index=False))

ValueError: 
All the 100 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "/Library/Framewo

In [ ]:
# Returns + average returns + standard deviation of returns (named volatility) for ONE file
folder = Path("DATA")
files = sorted(folder.glob("cleaned_*.csv"))

for file_path in files:
    df = pd.read_csv(file_path)

    # Optional: parse date if present
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("date").reset_index(drop=True)

    # Make sure close is numeric
    df["close"] = pd.to_numeric(df["close"], errors="coerce")

    # Daily returns from close price
    df["returns"] = df["close"].pct_change()

    # Drop NaN returns (first row and any bad rows)
    returns = df["returns"].dropna()

    # Required metrics
    avg_returns = returns.mean()
    volatility = returns.std()  # std deviation of returns = volatility

    print(f"File: {file_path}")
    print(f"Average Returns: {avg_returns:.6f}")
    print(f"Volatility (Std Dev of Returns): {volatility:.6f}")

    # Optional summary table
    summary_df = pd.DataFrame([{
        "file": file_path,
        "rows_used": len(df),
        "returns_count": len(returns),
        "avg_returns": avg_returns,
        "volatility": volatility
    }])

    display(summary_df)

File: DATA/cleaned_coin_Aave.csv
Average Returns: 0.010243
Volatility (Std Dev of Returns): 0.086548


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_Aave.csv,275,274,0.010243,0.086548


File: DATA/cleaned_coin_BinanceCoin.csv
Average Returns: 0.008493
Volatility (Std Dev of Returns): 0.080050


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_BinanceCoin.csv,1442,1441,0.008493,0.08005


File: DATA/cleaned_coin_Bitcoin.csv
Average Returns: 0.002741
Volatility (Std Dev of Returns): 0.042639


...

,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_Uniswap.csv,292,291,0.008031,0.091299


File: DATA/cleaned_coin_WrappedBitcoin.csv
Average Returns: 0.003512
Volatility (Std Dev of Returns): 0.042857


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_WrappedBitcoin.csv,888,887,0.003512,0.042857


File: DATA/cleaned_coin_XRP.csv
Average Returns: 0.004478
Volatility (Std Dev of Returns): 0.081566


,file,rows_used,returns_count,avg_returns,volatility
0,DATA/cleaned_coin_XRP.csv,2893,2892,0.004478,0.081566


file all


In [ ]:
# Logistic Regression across ALL cleaned files (linear terms only)
# Target: 1 if next day's Close > today's Close, else 0
# Train/Test: first 3/4 rows for training, last 1/4 rows for testing

folder = Path("DATA")
files = sorted(folder.glob("cleaned_*.csv"))

#to normalize the function like conveting to lower case 
def _norm(s: str):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def find_col(columns, candidates):
    norm_map = {_norm(c): c for c in columns}
    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]
    return None

base_features = ["open", "high", "low", "close", "volume"]

metrics_rows = []
weights_rows = []
feature_imp_rows = []
powers_rows = []
skipped_rows = []

for file_path in files:
    df = pd.read_csv(file_path)

    open_col  = find_col(df.columns, ["open", "priceopen"])
    high_col  = find_col(df.columns, ["high", "pricehigh"])
    low_col   = find_col(df.columns, ["low", "pricelow"])
    close_col = find_col(df.columns, ["close", "priceclose"])
    vol_col   = find_col(df.columns, ["volume", "volumefrom", "volumeto", "totalvolume", "tradevolume"])

    if any(c is None for c in [open_col, high_col, low_col, close_col, vol_col]):
        skipped_rows.append({
            "file": file_path.name,
            "reason": "missing one or more required columns"
        })
        continue

    data = df[[open_col, high_col, low_col, close_col, vol_col]].copy()
    data.columns = base_features
    data = data.apply(pd.to_numeric, errors="coerce")

    data["target_up_next_day"] = (data["close"].shift(-1) > data["close"]).astype(int)
    data = data.dropna().reset_index(drop=True)

    X = data[base_features]
    y = data["target_up_next_day"].astype(int)

    n = len(X)
    if n < 20:
        skipped_rows.append({
            "file": file_path.name,
            "reason": f"not enough usable rows ({n})"
        })
        continue

    split_idx = (3 * n) // 4
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, C=10.0, random_state=42))
    ])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    clf = model.named_steps["clf"]
    coef = clf.coef_.ravel()
    intercept = clf.intercept_[0]

    abs_coef = np.abs(coef)
    importance_pct = np.zeros_like(abs_coef) if abs_coef.sum() == 0 else (abs_coef / abs_coef.sum()) * 100

    feature_importance_df = pd.DataFrame({
        "feature": base_features,
        "weight": coef,
        "abs_weight": abs_coef,
        "importance_score_percent": importance_pct
    }).sort_values("importance_score_percent", ascending=False).reset_index(drop=True)
    feature_importance_df["rank"] = np.arange(1, len(feature_importance_df) + 1)

    metrics_rows.append({
        "file": file_path.name,
        "rows_used": n,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "accuracy": accuracy
    })

    w_row = {"file": file_path.name, "intercept": intercept}
    w_row.update({f"w_{f}": c for f, c in zip(base_features, coef)})
    weights_rows.append(w_row)

    powers_rows.append({
        "file": file_path.name,
        "feature": "intercept",
        "power_used": 0,
        "weight": intercept
    })
    for f_name, c in zip(base_features, coef):
        powers_rows.append({
            "file": file_path.name,
            "feature": f_name,
            "power_used": 1,
            "weight": c
        })

    for _, row in feature_importance_df.iterrows():
        feature_imp_rows.append({
            "file": file_path.name,
            "rank": int(row["rank"]),
            "feature": row["feature"],
            "weight": row["weight"],
            "abs_weight": row["abs_weight"],
            "importance_score_percent": row["importance_score_percent"]
        })

metrics_df = pd.DataFrame(metrics_rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
weights_df_all = pd.DataFrame(weights_rows)
powers_by_model_df = pd.DataFrame(powers_rows)
feature_importance_all_df = pd.DataFrame(feature_imp_rows)
skipped_df = pd.DataFrame(skipped_rows)

print("Model performance by file:")
display(metrics_df)

print("\nWeights by file:")
display(weights_df_all)

print("\nPowers used by each feature in each model:")
display(powers_by_model_df)

print("\nFeature importance by file:")
display(feature_importance_all_df)

if not skipped_df.empty:
    print("\nSkipped files:")
    display(skipped_df)

Model performance by file:


,file,rows_used,train_rows,test_rows,accuracy
0,cleaned_coin_Polkadot.csv,320,240,80,0.550000
1,cleaned_coin_WrappedBitcoin.csv,888,666,222,0.549550
2,cleaned_coin_Cardano.csv,1374,1030,344,0.537791
3,cleaned_coin_EOS.csv,1466,1099,367,0.528610
4,cleaned_coin_Bitcoin.csv,2991,2243,748,0.528075
5,cleaned_coin_ChainLink.csv,1385,1038,347,0.527378
6,cleaned_coin_Cosmos.csv,845,633,212,0.523585
7,cleaned_coin_Tether.csv,2318,1738,580,0.517241
8,cleaned_coin_Litecoin.csv,2991,2243,748,0.516043
9,cleaned_coin_Stellar.csv,2527,1895,632,0.515823



Weights by file:


,file,intercept,w_open,w_high,w_low,w_close,w_volume
0,cleaned_coin_Aave.csv,0.136720,-0.003835,-1.769417,0.137437,1.452201,0.161447
1,cleaned_coin_BinanceCoin.csv,0.047501,1.486461,-0.087218,-0.452687,-1.336303,0.290056
2,cleaned_coin_Bitcoin.csv,0.185283,0.309711,0.015150,-0.143575,-0.249485,0.067107
3,cleaned_coin_Cardano.csv,-0.005959,0.625918,0.924498,-2.208025,0.485421,0.012923
4,cleaned_coin_ChainLink.csv,-0.017166,1.381437,0.814582,-0.780043,-1.506473,0.188502
5,cleaned_coin_Cosmos.csv,0.007150,0.599560,-0.010713,0.635103,-1.387541,0.040283
6,cleaned_coin_CryptocomCoin.csv,-0.029948,-0.696010,-1.538664,2.697265,-0.237065,-0.142447
7,cleaned_coin_Dogecoin.csv,-0.146685,0.500327,0.608878,-0.965197,-0.220630,0.118007
8,cleaned_coin_EOS.csv,-0.042609,1.712957,-1.106939,-0.888133,0.250729,0.052507
9,cleaned_coin_Ethereum.csv,-0.044020,1.200361,0.586751,-0.738810,-1.069907,0.003092



Powers used by each feature in each model:


,file,feature,power_used,weight
0,cleaned_coin_Aave.csv,intercept,0,0.136720
1,cleaned_coin_Aave.csv,open,1,-0.003835
2,cleaned_coin_Aave.csv,high,1,-1.769417
3,cleaned_coin_Aave.csv,low,1,0.137437
4,cleaned_coin_Aave.csv,close,1,1.452201
...,...,...,...,...
133,cleaned_coin_XRP.csv,open,1,0.711547
134,cleaned_coin_XRP.csv,high,1,-0.066882
135,cleaned_coin_XRP.csv,low,1,-0.812440
136,cleaned_coin_XRP.csv,close,1,0.156250



Feature importance by file:


,file,rank,feature,weight,abs_weight,importance_score_percent
0,cleaned_coin_Aave.csv,1,high,-1.769417,1.769417,50.205664
1,cleaned_coin_Aave.csv,2,close,1.452201,1.452201,41.204949
2,cleaned_coin_Aave.csv,3,volume,0.161447,0.161447,4.580914
3,cleaned_coin_Aave.csv,4,low,0.137437,0.137437,3.899654
4,cleaned_coin_Aave.csv,5,open,-0.003835,0.003835,0.108819
...,...,...,...,...,...,...
110,cleaned_coin_XRP.csv,1,low,-0.812440,0.812440,46.286811
111,cleaned_coin_XRP.csv,2,open,0.711547,0.711547,40.538692
112,cleaned_coin_XRP.csv,3,close,0.156250,0.156250,8.901947
113,cleaned_coin_XRP.csv,4,high,-0.066882,0.066882,3.810415
